## Predicting Solar, Wind, Total Power Load, and Energy Price with the PyMC-BART Model

Here I will analyse an energy dataset provided by Kolasniwash on Kaggle to predict solar and wind power generation as well as power demand and price in Spain (see, <https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather/data>). For solar and wind power generation, I will predict each of these independent of each other while incorporating many different weather and time features. Next I will predict total power load on its own and then predict price using the predicted power load while also incorporating many different weather and time features. The ML model I will be using for all of the predicitons will be PyMC-BART, a Bayesian Additive Regression Trees non-parametric regression model.

In [ ]:
# Import the essential Python libraries we will definitely be using for this analysis.
import numpy as np
import pandas as pd
import os
import csv
from datetime import date, datetime, timezone          # To help deal with time series data.
import math
from tqdm import tqdm                  # Progress bar for loops that take a while.

# Import visualization libraries, I'm a big fan of Bokeh, but will also make use of Matplotlib.
# Matplotlib plotting library and functions.
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns

# Bokeh plotting library and functions. Note, this is probably an overkill but I have used most of these in my other projects.
from bokeh.io import output_notebook, show
from bokeh.models.annotations.labels import Label
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Whisker, BoxAnnotation, Arrow, OpenHead, Span
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import Legend, LinearAxis, Range1d, ColumnDataSource, LabelSet, HoverTool, DatetimeTickFormatter

import pymc as pm
import pymc_bart as pmb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from scipy.optimize import curve_fit
from loess.loess_1d import loess_1d
from bokeh.palettes import Category20
import pvlib
from pvlib.location import Location
import pgeocode
import holidays
from suntime import Sun
import lightgbm as lgb
from tqdm.auto import tqdm
import dill

In [ ]:
# Get current directory. I will use this to construct the file path to the data files.
current_dir = str(os.getcwd())

files_dir = current_dir + '/data/'

# dataframe holding power usage/generation data. Use default options unless we encounter issues.
power_pd = pd.read_csv(files_dir+'energy_dataset.csv', header=0)

# dataframe holding weather info. Use default options unless we encounter issues.
weather_pd = pd.read_csv(files_dir+'weather_features.csv', header=0)

In [ ]:
# Let's visualize power data by plotting power generation/demand for the various power sources along with
# some optional weather features.

# Set up plotting figure. Set x-axis to "datetime" so that the date time can be displayed appropriately.
def explore_plots(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black', color_features='black',
                  labels=None, features_labels=None, symbols=None, symbols_features=None, replaced_columns=None):

    """
    Function to create interactive exploratory plots.
    
    Parameters
    ----------
    dataframe : Pandas dataframe object
        The Pandas dataframe holding the data you want to plot.
    x_axis_column : string
        The name of the column you want to plot along the x-axis.
    y_axis_columns : list
        The names of the columns for the primary data you want to plot along the y-axis.
    x_axis_label : string
        The label to use for the x axis.
    y_axis_label : string
        The label to use for the y axis for the primary data. Note, all column data plotted will have the same y-axis scale
        and label name.
    title : string
        The title for the plot.

    Optional
    --------
    feature_columns : list or string
        The names of the columns you want to plot as additional features along secondary y-axis
        Default: None
    features_ylabel : list or string
        The secondary y-axis feature label/s. If a list greater than one item, list is concatenated into a single string.
        Default: None    
    p : bokeh plotting object
        A bokeh plotting object to add additional plotting objects to.
        Default: None
    normalize : Boolean
        Whether to normalize the data. Caution, might not work well for some features or when including more than one feature.
        Default: False
    other_colors : Boolean
        Whether to use your own colors (True) or to use color names as defined in this function (False).
        Default: False
    color_plot : string or list
        A string or list of colors to use for plotting the primary data. If list, must be length of number y_axis_columns.
        Default: 'black'
    color_features : string or list
        A string or list of colors to use for plotting the features data. If list, must be length of number feature_columns.
        Default: 'black'
    labels : list or string
        The labels to assign all the primary data from the y_axis_columns, should be list of length y_axis_columns if more than one
        primary data is being plotted.
        Default: None
    features_labels : list or string
        The labels to assign all the features data from feature_columns, should be list of length feature_columns if more than one
        feature is being plotted.
        Default: None
    symbols : string or list
        The plotting markers for the primary data. If list, must be length of number of y_axis_columns.
        Default: None
    symbols_features : string or list
        The plotting markers for the features data. If list, must be length of number of feature_columns.
        Default: None
    replaced_columns : Boolean
        Whether to plot the replaced primary data produced through cleaning. The appropriate columns must exist in the dataframe
        if True and given as 'replaced_' and the y_axis_columns, e.g., 'replaced_energy_usage_Wh' for replacements for 
        the 'energy_usage_Wh'.
        Default: False

    Returns
    -------
    p : bokeh object
        The Bokeh plotting object.
    """
    
    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}

    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        # Rotate labels for better readability
        p.xaxis.major_label_orientation = 120
        # Reduce the number of x-axis ticks to avoid crowding
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    # print('labels: ', labels)
    # print('markers: ', symbols)
    # print('colors: ', color_plot_values)
    
    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        # Create a loop to plot the column data for each city in Spain
        for y, city in enumerate(dataframe['city_name'].unique()):

            if normalize:
                max_primary_data = dataframe.loc[dataframe['city_name'] == city, column].max()
            else:
                max_primary_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 | y == 0 else False
            
            # Scatter plot with symbols.
            if symbols:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data, size=10,
                          marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                          legend_label=label + ' ' + str(city), visible=visible)
                
            # Add a line to connect the symbols
            p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                   dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data,
                   line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label + ' ' + str(city),
                   visible=visible)

            if replaced_columns:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, f'replaced_{column}']/max_primary_data,
                          size=10, color="red", marker="diamond", alpha=0.5, legend_label="Replaced " + label + ' ' + str(city),
                          visible=visible)

            # print('marker_index: ', marker_index)
            # print('color_index: ', color_index)
            # print('label_index: ', label_index)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
        label_index = label_index + 1
        

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        if normalize:
            max_features_data = dataframe[feature_columns].max().max()
        else:
            max_features_data = 1

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            # Create a loop to plot the features for each city.
            for y, city in enumerate(dataframe['city_name'].unique()):

                if normalize:
                    max_features_data = dataframe.loc[dataframe['city_name'] == city, feature_column].max()
                else:
                    max_features_data = 1

                # Only show the first column by default, hide others
                visible = True if i == 0 | y == 0 else False
                
                # Scatter plot with symbols. Only plot the energy usage as a function of time for a specific city.
                if symbols_features:
                    p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                              dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data,
                              y_range_name="features", size=10, marker=symbols_features[marker_index],
                              color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label + ' ' + str(city),
                              visible=visible)
                # Add a line to connect the symbols
                p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                       dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data, y_range_name="features",
                       line_width=2, color=color_plot_values[color_index], alpha=0.5,
                       legend_label=feature_label + ' ' + str(city), visible=visible)
            
                marker_index = marker_index + 1
                color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

In [ ]:
# Let's make copies of power_pd and weather_pd dataframes.
power_pd1 = power_pd.copy(deep=True)
weather_pd1 = weather_pd.copy(deep=True)

# Rename time columns for consistency in the power and weather dataframes.
power_pd1.rename(columns={'time':'local_time'}, inplace=True)
weather_pd1.rename(columns={'dt_iso':'local_time'}, inplace=True)

# Now convert the date/time strings in the local_time and utc_time columns to datetime64 data type so that I can work with
# the time series data.
power_pd1['local_time_tz'] = pd.to_datetime(power_pd1['local_time'], utc=True)
power_pd1['utc_time'] = power_pd1['local_time_tz'].dt.tz_localize(None)
power_pd1['local_time'] = power_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
power_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = power_pd1.pop('utc_time')
power_pd1.insert(1, 'utc_time', column_utc_time)

weather_pd1['local_time_tz'] = pd.to_datetime(weather_pd1['local_time'], utc=True)
weather_pd1['utc_time'] = weather_pd1['local_time_tz'].dt.tz_localize(None)
weather_pd1['local_time'] = weather_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
weather_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = weather_pd1.pop('utc_time')
weather_pd1.insert(1, 'utc_time', column_utc_time)

In [ ]:
# A column for city names in the power dataset and set to 'Spain' for now.
power_pd1['city_name'] = 'Spain'
column_city_name = power_pd1.pop('city_name')
power_pd1.insert(2, 'city_name', column_city_name)

In [ ]:
power_pd2 = power_pd1.copy(deep=True)

# Identify generation columns where all values are NaN and/or 0
null_zero = []
for col in power_pd2.columns:
    if col.startswith('generation'):
        # Create boolean mask where values are NaN or 0
        is_null_or_zero = power_pd2[col].isna() | (power_pd2[col] == 0.0)
        if is_null_or_zero.all():
            null_zero.append(col)

# Print and drop those columns
# print("Columns with all null/zero values:")
# for col in null_zero:
#     print(col)

# Drop from power_pd2
power_pd2.drop(columns=null_zero, inplace=True)
power_pd2.drop(columns=['forecast wind offshore eday ahead'], inplace=True)

In [ ]:
# Weather severity mapping that is used to help select the most severe weather conditions from two duplicate
# time stamps.

def get_weather_severity(weather_id):
    if 200 <= weather_id < 300:
        return 5  # Thunderstorm
    elif 300 <= weather_id < 400:
        return 3  # Drizzle
    elif 500 <= weather_id < 600:
        return 4  # Rain
    elif 600 <= weather_id < 700:
        return 4  # Snow
    elif 700 <= weather_id < 800:
        return 2  # Atmosphere (mist, fog)
    elif weather_id == 800:
        return 0  # Clear
    elif 801 <= weather_id <= 804:
        return 1  # Clouds
    else:
        return -1  # Unknown or invalid

In [ ]:
weather_pd2 = weather_pd1.copy(deep=True)

weather_pd2['severity'] = weather_pd2['weather_id'].apply(get_weather_severity)

# Sort by severity and weather_id
weather_pd2 = weather_pd2.sort_values(by=['city_name', 'utc_time', 'severity', 'weather_id'], ascending=[True, True, False, False])

# Identify the index of the rows that will be kept
keep_index = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').index

# Create the audit dataframe, mark rows as kept or dropped
weather_duplicates_audit = weather_pd2.copy(deep=True)
weather_duplicates_audit['keep_row'] = weather_duplicates_audit.index.isin(keep_index)

# Drop duplicates keeping most severe (and then highest ID) row
weather_pd2 = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').reset_index(drop=True)
weather_duplicates_audit.reset_index(drop=True, inplace=True)

# for city in weather_pd2['city_name'].unique():
#     print('Number of utc time rows in the weather dataset: ', len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time']),
#          ' in city: ', city)
#     print('Number of unique utc time rows in the weather dataset: ',
#           len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time'].unique()),
#          ' in city: ', city)
#     print(' ')

In [ ]:
from scipy.optimize import curve_fit

# Sine-like daily solar generation curve
def daily_solar_curve(hour, amplitude, phase_shift, vertical_shift):
    return amplitude * np.sin((np.pi / 12) * (hour - phase_shift)) + vertical_shift


def get_median_night_residual(df, date_col, hour_col, solar_col, target_date):
    """
    Returns the median nighttime value for the closest night to target_date in the dataframe.
    Nighttime is defined as hours < 6 or >= 18.
    """
    df_night = df[
        ((df[hour_col] < 6) | (df[hour_col] >= 18)) &
        (df[date_col].isin([target_date - pd.Timedelta(days=1), target_date, target_date + pd.Timedelta(days=1)]))
    ]
    # Select night closest to target_date
    night_dates = df_night[date_col].unique()
    if len(night_dates) == 0:
        return 0  # Fallback
    closest_night = min(night_dates, key=lambda d: abs((d - target_date).days))
    return df_night.loc[df_night[date_col] == closest_night, solar_col].median()

def fill_solar_with_rolling_fit(df, time_col, solar_col, filled_col='filled_generation solar',
                                replaced_col='replaced_generation_solar'):
    """
    Fills missing values in the solar generation column using a rolling 3-day window sinusoidal fit.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe containing the power generation data.
    time_col : str
        Name of the datetime column (must be datetime64).
    solar_col : str
        Name of the solar generation column.
    filled_col : str
        Name of the column where the filled (original + replacement) data is stored.
    replaced_col : str
        Name of the column where only the replacement values are stored.

    Returns
    ----------
    filled_data : pd.Series
        The column with the filled (original + replacement) data.
    replaced_data : pd.Series
        A column with only the replaced values (NaN elsewhere).
    """
    df = df.copy()
    df['hour'] = df[time_col].dt.hour
    df['date'] = df[time_col].dt.date
    filled_data = df[filled_col].copy()           # Column with the filled data
    replaced_data = df[replaced_col].copy()       # Column with the replaced data

    unique_dates = np.array(sorted(df['date'].unique()))
    is_daytime = lambda h: (h >= 6) & (h <= 18)

    # Iterate through center days for a 3-day rolling window
    for i in range(1, len(unique_dates) - 1):
        window_dates = unique_dates[i-1:i+2]
        center_day = unique_dates[i]

        mask_window = df['date'].isin(window_dates)
        window_df = df[mask_window & df[filled_col].notna()]
        window_df = window_df[is_daytime(window_df['hour'])]
        #window_df = window_df[(window_df['hour'] >= 6) & (window_df['hour'] <= 20)]

        if len(window_df) < 18:
            continue

        try:
            popt, _ = curve_fit(
                daily_solar_curve,
                window_df['hour'],
                window_df[filled_col],
                p0=[window_df[filled_col].max(), 12, 0],
                maxfev=10000
            )
        except Exception:
            continue

        mask_center = (
            (df['date'] == center_day) &
            (df[filled_col].isna()) &
            is_daytime(df['hour'])
        )

        # Identify consecutive NaN gaps (within the center day daytime)
        subidx = df[mask_center].index
        # We'll group gaps by consecutive indices (daytime NaNs)
        if not subidx.empty:
            groups = np.split(subidx, np.where(np.diff(subidx) != 1)[0]+1)
            for group in groups:
                if len(group) <= 2:
                    # Interpolate for short gaps
                    interp_vals = filled_data.interpolate(method='linear').loc[group]
                    filled_data.loc[group] = interp_vals
                    replaced_data.loc[group] = interp_vals
                else:
                    # Use curve fit for longer gaps
                    hours = df.loc[group, 'hour']
                    fitted_vals = daily_solar_curve(hours, *popt)
                    max_val = window_df[filled_col].max()
                    # Use median night value for negative fits
                    for idx, val, hr in zip(group, fitted_vals, hours):
                        if val < 0:
                            median_night_val = get_median_night_residual(df, 'date', 'hour', filled_col, center_day)
                            filled_data.loc[idx] = median_night_val
                            replaced_data.loc[idx] = median_night_val
                        else:
                            clipped_val = min(val, max_val)
                            filled_data.loc[idx] = clipped_val
                            replaced_data.loc[idx] = clipped_val

    return filled_data, replaced_data

In [ ]:
# Using the hybrid approach with local regression (LOESS) to fill in large gaps.
# I will use loess_1d function from loess as part of the Pypi library (https://pypi.org/project/loess/).
from loess.loess_1d import loess_1d

# Rolling average window size.
rolling_window = 5

# Make a deep copy of the dataframe in case of accidental modifications
power_pd_clean = power_pd2.copy(deep=True)

columns_to_clean = [
    'generation biomass',
    'generation fossil brown coal/lignite',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation fossil oil',
    'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage',
    'generation hydro water reservoir',
    'generation nuclear',
    'generation other',
    'generation other renewable',
    'generation solar',
    'generation waste',
    'generation wind onshore',
    'forecast solar day ahead',
    'forecast wind onshore day ahead',
    'total load forecast',
    'total load actual',
    'price day ahead',
    'price actual'
]

# Ensure datetime column is available as integer timestamps for LOESS
power_pd_clean['utc_timestamp'] = power_pd_clean['utc_time'].astype('int64') // 10**9

# Replace negative values with NaN (assumed invalid)
for col in columns_to_clean:
    power_pd_clean.loc[power_pd_clean[col] < 0, col] = np.nan

# Apply hybrid imputation to each column
for col in columns_to_clean:
    print(f"Processing column: {col}")
    
    # Create new columns to track replacement and filled values
    power_pd_clean[f'replaced_{col}'] = np.where(power_pd_clean[col].isna(), True, np.nan)
    power_pd_clean[f'filled_{col}'] = power_pd_clean[col].copy()

    #Need to handle solar power generation separately to the other columns as the LOESS approach does not work well.
    if col == 'generation solar':
        # Use the fill_solar_with_rolling_fit function to fill the missing values for the solar generation column.
        solar_filled, solar_replaced = fill_solar_with_rolling_fit(power_pd_clean, 'local_time', 'generation solar',
                                                                   filled_col='filled_generation solar',
                                                                   replaced_col='replaced_generation solar')
        power_pd_clean[f'filled_{col}'] = solar_filled
        power_pd_clean[f'replaced_{col}'] = solar_replaced
        
    else:
        # Step 1: Rolling average for small gaps
        rolling_filled = power_pd_clean[col].rolling(window=rolling_window, center=True, min_periods=1).mean()
        small_gap_mask = power_pd_clean[col].isna()
        power_pd_clean.loc[small_gap_mask, f'filled_{col}'] = rolling_filled[small_gap_mask]
        power_pd_clean.loc[small_gap_mask, f'replaced_{col}'] = rolling_filled[small_gap_mask]
      
        # Step 2: LOESS for medium gaps
        subset = power_pd_clean.copy()
        subset['utc_timestamp'] = power_pd_clean['utc_timestamp']
        mask_loess = subset[f'filled_{col}'].isna()
    
        valid_idx = subset[col].notna()
        missing_idx = subset[col].isna()
        loess_frac = 0.0005
    
        if valid_idx.sum() > 10 and missing_idx.sum() > 0:
            valid_timestamps = subset.loc[valid_idx, 'utc_timestamp'].to_numpy()
            valid_values = subset.loc[valid_idx, col].to_numpy()
            missing_timestamps = subset.loc[missing_idx, 'utc_timestamp'].to_numpy()
    
            #print(missing_timestamps)
    
            try:
                time_out, loess_fitted, _ = loess_1d(valid_timestamps, valid_values, xnew=missing_timestamps, frac=loess_frac)
    
                if len(loess_fitted) > 0:
                    loess_df = pd.DataFrame({'utc_timestamp': missing_timestamps, 'loess_value': loess_fitted})
                    loess_df.set_index(subset.loc[missing_idx].index, inplace=True)
        
                    power_pd_clean.loc[mask_loess, f'filled_{col}'] = loess_df['loess_value']
                    power_pd_clean.loc[mask_loess, f'replaced_{col}'] = loess_df['loess_value']
    
            except Exception as e:
                print(f"⚠️ LOESS failed for column '{col}': {e}")
                # Fallback to linear interpolation
                interpolated_values = power_pd_clean[col].interpolate(method='linear')
                power_pd_clean.loc[mask_loess, f'filled_{col}'] = interpolated_values[mask_loess]
                power_pd_clean.loc[mask_loess, f'replaced_{col}'] = interpolated_values[mask_loess]

    # Step 3: Final fallback — linear interpolation
    final_mask = power_pd_clean[f'filled_{col}'].isna()
    power_pd_clean.loc[final_mask, f'filled_{col}'] = power_pd_clean[col].interpolate(method='linear')[final_mask]
    power_pd_clean.loc[final_mask, f'replaced_{col}'] = power_pd_clean[f'filled_{col}'][final_mask]

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_dataset_missing_LOESS_replaced2.csv'
power_pd_clean.to_csv(filename_out, index=False)

In [ ]:
columns_final = ['local_time', 'utc_time', 'utc_timestamp', 'city_name',  
       'filled_generation biomass',
       'filled_generation fossil brown coal/lignite',
       'filled_generation fossil gas',
       'filled_generation fossil hard coal',
       'filled_generation fossil oil',
       'filled_generation hydro pumped storage consumption',
       'filled_generation hydro run-of-river and poundage',
       'filled_generation hydro water reservoir',
       'filled_generation nuclear',
       'filled_generation other',
       'filled_generation other renewable',
       'filled_generation solar',
       'filled_generation waste',
       'filled_generation wind onshore',
       'filled_forecast solar day ahead',
       'filled_forecast wind onshore day ahead',
       'filled_total load forecast',
       'filled_total load actual',
       'filled_price day ahead',
       'filled_price actual']

power_pd_final = power_pd_clean[columns_final].copy(deep=True)

filled_cols = [
    'filled_generation biomass',
    'filled_generation fossil brown coal/lignite',
    'filled_generation fossil gas',
    'filled_generation fossil hard coal',
    'filled_generation fossil oil',
    'filled_generation hydro pumped storage consumption',
    'filled_generation hydro run-of-river and poundage',
    'filled_generation hydro water reservoir',
    'filled_generation nuclear',
    'filled_generation other',
    'filled_generation other renewable',
    'filled_generation solar',
    'filled_generation waste',
    'filled_generation wind onshore',
    'filled_forecast solar day ahead',
    'filled_forecast wind onshore day ahead',
    'filled_total load forecast',
    'filled_total load actual',
    'filled_price day ahead',
    'filled_price actual'
]

rename_dict = {col: col.replace('filled_', '') for col in filled_cols}

power_pd_final.rename(columns=rename_dict, inplace=True)

In [ ]:
merged_power_weather = pd.merge(weather_pd2, power_pd_final, on='utc_time', how='left')
merged_power_weather.drop(columns=['local_time_y', 'city_name_y'], axis=1, inplace=True)
merged_power_weather.rename(columns={'local_time_x': 'local_time', 'city_name_x': 'city_name'}, inplace=True)

In [ ]:
# Now I'll create a function to compute the solar irradiance as a function of local time and city.
import pvlib
from pvlib.location import Location
import pgeocode

def calculate_solar_flux(row):
    location = pvlib_locs[row['city_name']]
    # Get the position of the Sun, specifically the zenith elevation.
    solpos = location.get_solarposition(row['utc_time'])
    # Get the day of the year which is used for calculating Sun's position.
    day_of_year = row['local_time'].dayofyear
    # Use 1361Wm^2  as the solar radiation 'constant'.
    dni = pvlib.irradiance.disc(1361.0, solpos['zenith'], day_of_year)['dni']
    return dni.item()     # Convert single element numpy array to float when returning

In [ ]:
merged_power_weather_feat = merged_power_weather.copy(deep=True)

# tqdm progress bar for pandas operations.
tqdm.pandas()

# Precompute city coordinate and put in dictionary.
city_names = merged_power_weather_feat['city_name'].unique()
nomi = pgeocode.Nominatim('ES')
city_coords = {}

for city in city_names:
    result = nomi.query_location(city)
    # Take the first row (if Series, convert; if DataFrame, take .iloc[0])
    if hasattr(result, 'iloc'):
        lat = float(result.iloc[0]['latitude'])
        lon = float(result.iloc[0]['longitude'])
    else:  # If already Series
        lat = float(result['latitude'])
        lon = float(result['longitude'])
    city_coords[city] = (lat, lon)

pvlib_locs = {city: Location(lat, lon) for city, (lat, lon) in city_coords.items()}

In [ ]:
# Calculate the solar flux using pvlib and pgeocode by calling the calculate_solar_flux function I created in the previous cell.
# Instead of the normal .apply Pandas operation, I'll using progress_apply as the following takes some time (about 30 minutes).
merged_power_weather_feat['solar_flux_Wm2'] = merged_power_weather_feat.progress_apply(calculate_solar_flux, axis=1)

In [ ]:
# Let's add the additional time/date features.
import holidays
from suntime import Sun
from datetime import date, datetime, timezone

# Compute the meteorological season based on month
def get_season(month):
    if month in [3, 4, 5]:
        # Spring
        return 1
    elif month in [6, 7, 8]:
        # Summer
        return 2
    elif month in [9, 10, 11]:
        # Autumn
        return 3
    else:
        # Winter
        return 0

# Determine the sunrise and sunset times.
def precompute_sunrise_sunset(city_coords_dict, all_dates):
    records = []
    for city, (lat, lon) in city_coords_dict.items():
        sun = Sun(lat, lon)
        for d in all_dates:
            # Make sure d is a datetime.date object (not pd.Timestamp)
            if isinstance(d, pd.Timestamp):
                d_py = d.date()
            else:
                d_py = d
            dt = datetime.combine(d_py, datetime.min.time()).replace(tzinfo=timezone.utc)
            try:
                sunrise = sun.get_sunrise_time(dt)
                sunset = sun.get_sunset_time(dt)
            except Exception as e:
                print(f"Failed for {city} {d_py}: {e}")
                sunrise, sunset = pd.NaT, pd.NaT
            records.append({'city_name': city, 'date': d_py, 'sunrise_time_utc': sunrise, 'sunset_time_utc': sunset})
    return pd.DataFrame(records)

def add_time_features(df, city_coords_dict, sun_table=None):
    df = df.copy()

    df['date'] = df['utc_time'].dt.date
    df['month'] = df['utc_time'].dt.month

    # If no precomputed table is given, build it now:
    if sun_table is None:
        all_dates = df['date'].unique()
        sun_table = precompute_sunrise_sunset(city_coords_dict, all_dates)

    # Merge sun_table (on city and date) into main dataframe
    df = df.merge(sun_table, on=['city_name', 'date'], how='left')

    # Remove timezone info for naive datetime columns (removes +00:00)
    df['sunrise_time_utc_naive'] = df['sunrise_time_utc'].dt.tz_localize(None)
    df['sunset_time_utc_naive'] = df['sunset_time_utc'].dt.tz_localize(None)

    # Daylight length (in hours)
    df['daylight_length_hr'] = (df['sunset_time_utc_naive'] - df['sunrise_time_utc_naive']).dt.total_seconds() / 3600.0

    neg_mask = df['daylight_length_hr'] < 0
    df.loc[neg_mask, 'daylight_length_hr'] = df.loc[neg_mask, 'daylight_length_hr'] + 24.0

    df['sunrise_time_utc'] = df['sunrise_time_utc_naive'].dt.time
    df['sunset_time_utc'] = df['sunset_time_utc_naive'].dt.time

    # Sunrise/sunset as fractions of day (UTC)
    df['sunrise_day_fraction_utc'] = (
        df['sunrise_time_utc_naive'].dt.hour / 24 +
        df['sunrise_time_utc_naive'].dt.minute / (24*60) +
        df['sunrise_time_utc_naive'].dt.second / (24*3600)
    )
    df['sunset_day_fraction_utc'] = (
        df['sunset_time_utc_naive'].dt.hour / 24 +
        df['sunset_time_utc_naive'].dt.minute / (24*60) +
        df['sunset_time_utc_naive'].dt.second / (24*3600)
    )

    df = df.drop(columns=['sunrise_time_utc_naive', 'sunset_time_utc_naive'])

    # UTC time as fraction of day
    df['utc_time_day_fraction'] = df['utc_time'].dt.hour/24.0 + df['utc_time'].dt.minute/(24.0*60) + df['utc_time'].dt.second/(24.0*3600)
    # Date as fraction of year
    df['day_of_year'] = df['utc_time'].dt.dayofyear
    df['date_fraction_of_year'] = df['day_of_year'] / 365.25

    # Day of week (Monday=0)
    df['day_of_week'] = df['utc_time'].dt.dayofweek
    # # Is weekend (0 for Mon–Fri, 1 for Sat/Sun)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Cyclical time encodings
    df['day_hour_sin'] = np.sin(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_hour_cos'] = np.cos(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['season'] = df['month'].apply(get_season)

    years = df['local_time'].dt.year.unique().tolist()
    es_holidays = holidays.country_holidays('ES', years=years)
    df['is_holiday'] = df['local_time'].dt.date.isin(es_holidays).astype(int)

    return df

In [ ]:
merged_power_weather_feat2 = merged_power_weather_feat.copy(deep=True)

# Precompute sunrise/sunset for all unique dates and cities
all_dates = merged_power_weather_feat2['utc_time'].dt.date.unique()
sun_table = precompute_sunrise_sunset(city_coords, all_dates)

merged_power_weather_feat2 = add_time_features(merged_power_weather_feat2, city_coords, sun_table)
#merged_power_weather_feat2 = add_time_features(merged_power_weather_feat2, city_coords)

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_weather_dataset_with_new_features.csv'
merged_power_weather_feat2.to_csv(filename_out, index=False)

In [ ]:
merged_power_weather_feat_final = merged_power_weather_feat2.copy(deep=True)

# Weather columns to roll by city
weather_features = [
    'temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
    'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'solar_flux_Wm2'
]

# Power features to roll (national)
power_features = [
    'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
    'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation nuclear', 'generation other', 'generation other renewable', 'generation solar',
    'generation waste', 'generation wind onshore'
]

# Features to roll for both 3h and 6h (e.g., for load and price)
special_features = ['total load actual', 'price actual']

In [ ]:
# WEATHER: by city_name and sorted by utc_time
merged_power_weather_feat_final = merged_power_weather_feat_final.sort_values(['city_name', 'utc_time'])
for col in weather_features:

    # Need to include shift(1) to ensure no future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final
        .groupby('city_name')[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )

    # Fill the first row of each city (where shift(1) makes it NaN) with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

# POWER: National level (no city), sorted by utc_time
merged_power_weather_feat_final = merged_power_weather_feat_final.sort_values('utc_time')
for col in power_features:

    # Need to include shift(1) to ensure no future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=3, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

# Special features (total load actual, price actual): 3h and 6h
for col in special_features:

    # Need to include shift(1) to ensure no future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=3, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

    # Need to include shift(1) to ensure no future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling6h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=6, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling6h'] = (
        merged_power_weather_feat_final[f'{col}_rolling6h']
        .fillna(merged_power_weather_feat_final[col])
    )

In [ ]:
time_feature_columns = [
    'date', 'month', 'sunrise_time_utc', 'sunset_time_utc', 'daylight_length_hr', 'sunrise_day_fraction_utc',
    'sunset_day_fraction_utc', 'utc_time_day_fraction', 'day_of_year', 'date_fraction_of_year',
    'day_of_week', 'is_weekend', 'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos',
    'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday'
]

# Remove from current columns and re-insert at the right spot
cols = merged_power_weather_feat_final.columns.tolist()
# Remove time_feature_columns from their current positions
cols = [c for c in cols if c not in time_feature_columns]
# Insert after 'utc_time'
utc_time_idx = cols.index('utc_time')
for offset, c in enumerate(time_feature_columns):
    cols.insert(utc_time_idx + 1 + offset, c)

In [ ]:
def insert_rolling_after_original(cols, df, rolling_suffixes=['_rolling3h', '_rolling6h']):
    new_cols = []
    already_added = set()
    for c in cols:
        new_cols.append(c)
        already_added.add(c)
        # Insert rolling variant(s) if present
        for sfx in rolling_suffixes:
            rc = c + sfx
            if rc in df.columns and rc not in already_added:
                new_cols.append(rc)
                already_added.add(rc)
    # Do not append remaining rolling columns again!
    return new_cols

# Rebuild cols to avoid duplication
cols = merged_power_weather_feat_final.columns.tolist()

# Remove the rolling columns from their current position (if present)
for col in merged_power_weather_feat_final.columns:
    if col.endswith('_rolling3h') or col.endswith('_rolling6h'):
        if col in cols:
            cols.remove(col)

# Insert rolling columns next to originals, without duplication
cols = insert_rolling_after_original(cols, merged_power_weather_feat_final)
merged_power_weather_feat_final = merged_power_weather_feat_final[cols]

In [ ]:
# 1. Main time columns
main_time_cols = ['local_time', 'utc_time', 'utc_timestamp']

# 2. Time feature columns
time_feature_cols = [
    'date', 'month', 'sunrise_time_utc', 'sunset_time_utc', 'daylight_length_hr',
    'sunrise_day_fraction_utc', 'sunset_day_fraction_utc', 'utc_time_day_fraction',
    'day_of_year', 'date_fraction_of_year', 'day_of_week', 'is_weekend', 'day_hour_sin',
    'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin',
    'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday'
]

# 3. City column
city_col = ['city_name']

# 4. The rest: everything except above (order: weather, power, forecasts, rolling, etc.)
exclude = set(main_time_cols + time_feature_cols + city_col)
rest_cols = [c for c in merged_power_weather_feat_final.columns if c not in exclude]

# To get rolling columns next to original, do this:
def get_original_and_rolling(cols):
    used = set()
    new_cols = []
    for col in cols:
        if col in used:
            continue
        new_cols.append(col)
        used.add(col)
        # Add matching rolling cols
        for suf in ['_rolling3h', '_rolling6h']:
            rolling_col = f"{col}{suf}"
            if rolling_col in cols and rolling_col not in used:
                new_cols.append(rolling_col)
                used.add(rolling_col)
    return new_cols

# Order weather and power columns with rolling features next to originals
weather_power_rolling_cols = [
    'temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
    'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'solar_flux_Wm2', 'weather_id', 'weather_main',
    'weather_description', 'weather_icon', 'severity', 
    'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
    'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation nuclear', 'generation other', 'generation other renewable',
    'generation solar', 'generation waste', 'generation wind onshore',
    'total load actual', 'price actual'
]

# Add rolling for weather/power columns
weather_power_rolling_cols_all = []
for col in weather_power_rolling_cols:
    weather_power_rolling_cols_all.append(col)
    if f"{col}_rolling3h" in merged_power_weather_feat_final.columns:
        weather_power_rolling_cols_all.append(f"{col}_rolling3h")
    if f"{col}_rolling6h" in merged_power_weather_feat_final.columns:
        weather_power_rolling_cols_all.append(f"{col}_rolling6h")

# Other columns (forecast, weather description, etc.) in their original order
leftover_cols = [c for c in rest_cols if c not in weather_power_rolling_cols_all]

# Final column order
final_order = (
    main_time_cols +
    time_feature_cols +
    city_col +
    weather_power_rolling_cols_all +
    leftover_cols
)

# Reorder the dataframe
merged_power_weather_feat_final = merged_power_weather_feat_final[final_order]

In [ ]:
merged_power_weather_feat_final['city_name'] = merged_power_weather_feat_final['city_name'].str.strip()

city_populations_2016 = {
    'Madrid': 3238048,
    'Barcelona': 1619286,
    'Valencia': 790448,
    'Seville': 691191,
    'Bilbao': 348553
}

total_population = sum(city_populations_2016.values())
city_weights = {city: pop / total_population for city, pop in city_populations_2016.items()}

merged_power_weather_feat_final['city_weights'] = merged_power_weather_feat_final['city_name'].map(city_weights)

In [ ]:
# I'll start with predicting solar generation.

# --- Ensure your dataframe is sorted ---
df = merged_power_weather_feat_final.copy()
df = df.sort_values('utc_time')

target_col = 'generation solar'
forecast_col = 'forecast solar day ahead'
datetime_col = 'utc_time'
city_col = 'city_name'

# Exclude categorical weather, non-numeric, and other power columns. The columns that are kept have been specified so this really
# isn't necessary, but might be useful in the future.
exclude_cols = ['local_time', 'utc_time', 'utc_timestamp', 'date', 'weather_main', 'weather_description',
                'weather_icon', 'generation solar', 'generation solar_rolling3h', 'generation wind onshore',
                'generation wind onshore_rolling3h', 'generation biomass', 'generation biomass_rolling3h',
                'generation fossil brown coal/lignite', 'generation fossil brown coal/lignite_rolling3h',
                'generation fossil gas', 'generation fossil gas_rolling3h',
                'generation fossil hard coal', 'generation fossil hard coal_rolling3h',
                'generation fossil oil', 'generation fossil oil_rolling3h',
                'generation hydro pumped storage consumption',
                'generation hydro pumped storage consumption_rolling3h',
                'generation hydro run-of-river and poundage',
                'generation hydro run-of-river and poundage_rolling3h',
                'generation hydro water reservoir',
                'generation hydro water reservoir_rolling3h', 'generation nuclear',
                'generation nuclear_rolling3h', 'generation other',
                'generation other_rolling3h', 'generation other renewable',
                'generation other renewable_rolling3h', 'generation waste',
                'generation waste_rolling3h', 'total load actual', 'total load actual_rolling3h',
                'total load actual_rolling6h', 'price actual', 'price actual_rolling3h', 'price actual_rolling6h',
                'forecast solar day ahead', 'forecast wind onshore day ahead',
                'total load forecast', 'price day ahead', 'city_name']

# Specify columns to weight based on population.
features_to_weight = ['daylight_length_hr', 'sunrise_day_fraction_utc', 'sunset_day_fraction_utc', 'temp',
                      'temp_rolling3h', 'temp_min', 'temp_min_rolling3h', 'temp_max', 'temp_max_rolling3h',
                      'pressure', 'pressure_rolling3h', 'humidity', 'humidity_rolling3h', 'wind_speed',
                      'wind_speed_rolling3h', 'wind_deg', 'wind_deg_rolling3h', 'rain_1h', 'rain_1h_rolling3h',
                      'rain_3h', 'rain_3h_rolling3h', 'snow_3h', 'snow_3h_rolling3h', 'clouds_all', 'clouds_all_rolling3h',
                      'solar_flux_Wm2', 'solar_flux_Wm2_rolling3h', 'severity']

# Don't apply the city weights to these columns as they will be the same for all cities.
time_features_no_weight = ['month', 'day_of_year', 'date_fraction_of_year', 'day_of_week',
                           'is_weekend', 'season', 'is_holiday', 'utc_time_day_fraction',
                           'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos',
                           'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos']

print("Features used for LightGBM:", features_to_weight + time_features_no_weight)

# Apply the city weights (based on population) to aggregate the weather/relevant time features across the cities.
agg_df = (
    df.groupby('utc_time', group_keys=False)
      .agg({col: lambda x: np.sum(x * df.loc[x.index, 'city_weights']) for col in features_to_weight})
)

time_feats = df.groupby('utc_time', group_keys=False)[time_features_no_weight].first()
agg_df = agg_df.join(time_feats)

# Add the national targets & forecast values
agg_df[target_col] = df.groupby('utc_time')[target_col].first()
agg_df[forecast_col] = df.groupby('utc_time')[forecast_col].first()
agg_df = agg_df.reset_index()
agg_df = agg_df.sort_values('utc_time').reset_index(drop=True)

# df is the pre-aggregated per-city dataframe (before agg_df), with columns: utc_time, city_name, city_weights, weather_main
tmp = df[['utc_time', 'city_weights', 'weather_main']].copy()
onehot = pd.get_dummies(tmp['weather_main'], prefix='wx')
weighted = onehot.mul(tmp['city_weights'].values, axis=0)

wx_dist = pd.concat([tmp[['utc_time']], weighted], axis=1) \
             .groupby('utc_time', as_index=True).sum()

# Normalize to sum to 1 in case of missing cities:
wx_dist = wx_dist.div(wx_dist.sum(axis=1), axis=0).fillna(0.0)

# Merge into your aggregated feature table
agg_df = agg_df.join(wx_dist, on='utc_time')

In [ ]:
direct_out = current_dir + '/output'
filename_out = direct_out + '/agreggated_dataset_column_stats.csv'
agg_df.describe().to_csv(filename_out, index=True)